In [1]:
# for the MACE MD using ASE ######

import numpy as np
import ase.io
from ase.calculators.espresso import Espresso
import os


sample_name = 'GO-40-1-nvt-mace'
#sample_name = 'GO-30-oh-o-3-nvt-mace'
strain_path = "../4-mace-finetune/mace-training-12/md/40-1/strain_GO-40-1.dat"
traj_path = "../4-mace-finetune/mace-training-12/md/40-1/dump_nvt_GO-40-1.xyz"
folder = "../4-mace-finetune/mace-training-12/md/40-1/"
out_dir = os.path.join(folder, f"extracted-config_nvt{sample_name}")
os.makedirs(out_dir, exist_ok=True)



# =============================================================================================
#  Sampling setup based on strain
# I've considered for the 30 %, the region shall be changed based on oxidation!
# ===============================================================================================
regions = {
    "nvt":  {"low": 0.00, "high": 0.117250, "n": 150},
    "plastic":  {"low": 0.05, "high": 0.12, "n": 60},
    "break":    {"low": 0.12, "high": 0.2, "n": 20},
}


# =============================================================================================
#  QE & pseudopotentials                                        
# =============================================================================================
pseudopotentials = {
    'H': 'H.pbe-rrkjus_psl.1.0.0.UPF', 
    'C': 'C.pbe-n-rrkjus_psl.1.0.0.UPF',
    'O': 'O.pbe-n-rrkjus_psl.1.0.0.UPF',
}
    'pseudo_dir': '/home/sara/qe/pseudo',
   # 'pseudo_dir': '/home/cicn/cicn864958/qe/pseudo',
input_qe_std = {
    'calculation': 'scf',
    'outdir': '.',
    'pseudo_dir': '/home/cicn/cicn864958/qe/pseudo',
    'disk_io': 'none',
    'system': {
        'ecutwfc': 90,
        'nosym': True,
        'input_dft': 'PBE',
        'occupations': 'smearing',
        'smearing': 'fermi-dirac',
        'degauss': 0.001,
        'ibrav': 0,
        'nspin': 1,
    },
    'electrons': {
        'mixing_beta': 0.3,
        'electron_maxstep': 1000,
        'mixing_mode': 'plain',
        'conv_thr': 1e-8,
        'diagonalization': 'david',
    },
}


# =====================================================
# LOAD DATA

strain_vc = np.loadtxt(strain_path)
steps = strain_vc[:,0]
strain = strain_vc[:,1]
dump_interval = 100

traj = ase.io.read(traj_path, format="extxyz", index=":")
n_frames = len(traj)
print(f" The {len(traj)} frames from trajectory is loaded.")

frame_strain = []

for i in range(n_frames):
    md_step = i * dump_interval
    # find strain corresponding to this MD step
    idx = np.argmin(abs(steps - md_step))
    frame_strain.append(strain[idx])

frame_strain = np.array(frame_strain)


# =============================================================================================
#  Sampling by region 

counter = 1  

for phase, info in regions.items():
    low, high, n_pick = info["low"], info["high"], info["n"]
    
    idx = np.where((frame_strain >= low) & (frame_strain < high))[0]
    
    if len(idx) == 0:
        print(f"WARNING: no frames found for phase {phase}")
        continue

    pick_idx = np.linspace(0, len(idx)-1, n_pick, dtype=int)
    selected_frames = idx[pick_idx]

    print(f"Phase '{phase}': found {len(idx)} frames → selecting {n_pick}")

    for fi in selected_frames:
        conf = traj[fi]

        fname = os.path.join(out_dir, f"pw-{sample_name}-{phase}{counter:03d}.in")
    
        ase.io.write(fname, conf,
                 format="espresso-in",
                 input_data=input_qe_std,
                 pseudopotentials=pseudopotentials,
                 tprnfor=True) 
        print(f"  → wrote {fname}   (strain={frame_strain[fi]:.4f})")
        counter += 1

print(f"\n Extracting is done! Total QE inputs written: {counter-1}")

 The 3801 frames from trajectory is loaded.
Phase 'nvt': found 2345 frames → selecting 150
  → wrote ../4-mace-finetune/mace-training-12/md/40-1/extracted-config_nvtGO-40-1-nvt-mace-ActiveL-second/pw-GO-40-1-nvt-mace-ActiveL-second-nvt001.in   (strain=0.0000)
  → wrote ../4-mace-finetune/mace-training-12/md/40-1/extracted-config_nvtGO-40-1-nvt-mace-ActiveL-second/pw-GO-40-1-nvt-mace-ActiveL-second-nvt002.in   (strain=0.0008)
  → wrote ../4-mace-finetune/mace-training-12/md/40-1/extracted-config_nvtGO-40-1-nvt-mace-ActiveL-second/pw-GO-40-1-nvt-mace-ActiveL-second-nvt003.in   (strain=0.0016)
  → wrote ../4-mace-finetune/mace-training-12/md/40-1/extracted-config_nvtGO-40-1-nvt-mace-ActiveL-second/pw-GO-40-1-nvt-mace-ActiveL-second-nvt004.in   (strain=0.0024)
  → wrote ../4-mace-finetune/mace-training-12/md/40-1/extracted-config_nvtGO-40-1-nvt-mace-ActiveL-second/pw-GO-40-1-nvt-mace-ActiveL-second-nvt005.in   (strain=0.0031)
  → wrote ../4-mace-finetune/mace-training-12/md/40-1/extracted-